# Optimización del Modelo Seleccionado: Regresión Logística

En este notebook se optimiza el modelo ganador identificado en el análisis comparativo: **Regresión Logística (sin balance)**.

El objetivo es mejorar su rendimiento mediante búsqueda de hiperparámetros, validación cruzada y evaluación cuantitativa y cualitativa, siguiendo las recomendaciones del Proyecto Final:

- Optimización precisa (GridSearch / RandomSearch)
- Evaluación de criterios cuantitativos (F1-score, AUC)
- Evaluación de criterios inadecuados (tiempo, interpretabilidad, simplicidad)
- Documentación de trade-offs


# Carga de datos y procesamiento

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset limpio final
df = pd.read_parquet("data/processed/survey_ai_usage_clean_filtered.parquet")

df.head()


,DevType,WorkExp,LanguageHaveWorkedWith,Country,RemoteWork,Industry,OrgSize,EdLevel,AI_Usage,NumLanguages
0,"Developer, mobile",8.0,Bash/Shell (all shells);Dart;SQL,Ukraine,Remote,Fintech,20 to 99 employees,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",1,3
1,"Developer, back-end",2.0,Java,Netherlands,"Hybrid (some in-person, leans heavy to flexibi...",Retail and Consumer Services,500 to 999 employees,"Associate degree (A.A., A.S., etc.)",1,1
2,"Developer, front-end",10.0,Dart;HTML/CSS;JavaScript;TypeScript,Ukraine,No especificado,Software Development,No especificado,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",1,4
3,"Developer, back-end",4.0,Java;Kotlin;SQL,Ukraine,Remote,Retail and Consumer Services,"10,000 or more employees","Bachelor’s degree (B.A., B.S., B.Eng., etc.)",1,3
4,Engineering manager,21.0,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,Ukraine,No especificado,Software Development,No especificado,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",1,14


# Separacion de variables

In [ ]:
X = df.drop(columns=["AI_Usage"])
y = df["AI_Usage"]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols)
])


# Modelo base (antes de optimización)

In [ ]:
base_model = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(max_iter=500))
])

base_model.fit(X, y)

y_pred = base_model.predict(X)

print("F1-score base:", f1_score(y, y_pred))
print("AUC base:", roc_auc_score(y, base_model.predict_proba(X)[:,1]))


F1-score base: 0.8933883869184389
AUC base: 0.8404199306779726


# Búsqueda de Hiperparámetros (Grid Search)

In [ ]:
param_grid = {
    "clf__penalty": ["l1", "l2", "elasticnet", None],
    "clf__C": [0.001, 0.01, 0.1, 1, 10],
    "clf__solver": ["lbfgs", "liblinear", "saga"],
    "clf__l1_ratio": [0, 0.2, 0.5, 0.8, 1]  # aplica solo si elasticnet
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    base_model,
    param_grid,
    cv=cv,
    scoring="f1",
    verbose=1,
    n_jobs=-1
)

grid.fit(X, y)

grid.best_params_, grid.best_score_


Fitting 5 folds for each of 300 candidates, totalling 1500 fits


# Random Search (más rápido y amplio)

In [ ]:
from scipy.stats import uniform, randint

param_dist = {
    "clf__penalty": ["l1", "l2", "elasticnet"],
    "clf__C": uniform(0.001, 10),
    "clf__l1_ratio": uniform(0, 1),
    "clf__solver": ["saga"]
}

random_search = RandomizedSearchCV(
    base_model,
    param_dist,
    n_iter=30,
    cv=cv,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

random_search.best_params_, random_search.best_score_


# Comparación entre Modelo Base y Optimizado

In [ ]:
results = pd.DataFrame({
    "Modelo": ["Base", "Optimizado"],
    "F1": [f1_score(y, y_pred),
           random_search.best_score_],
})

results["Mejora_%"] = (results["F1"] - results["F1"].iloc[0]) / results["F1"].iloc[0] * 100
results


# Evaluación Final del Modelo Optimizado

In [ ]:
best_model = random_search.best_estimator_

y_pred_opt = best_model.predict(X)
y_prob_opt = best_model.predict_proba(X)[:,1]

print(classification_report(y, y_pred_opt))
print("AUC optimizado:", roc_auc_score(y, y_prob_opt))


# Graficar Matriz de Confusión

In [ ]:
cm = confusion_matrix(y, y_pred_opt)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusión - Logistic Regression Optimizado")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()


# Curva ROC del Modelo Optimizado

In [ ]:
fpr, tpr, _ = roc_curve(y, y_prob_opt)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"Optimizado (AUC={roc_auc_score(y, y_prob_opt):.3f})")
plt.plot([0,1],[0,1],"k--", label="Aleatorio")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Modelo Optimizado")
plt.legend()
plt.show()


# Criterios Inadecuados Evaluados

## Evaluación de Criterios Cualitativos (Inadecuados)

- **Tiempo de entrenamiento:** bajo (LR es muy rápida).
- **Tiempo de inferencia:** extremadamente bajo.
- **Interpretabilidad:** excelente (coeficientes accesibles).
- **Simplicidad:** el modelo es ligero y fácil de desplegar.
- **Escalabilidad:** alto rendimiento en grandes volúmenes de datos.

El modelo optimizado mantiene todas estas ventajas.


# Conclusión de la Optimización

- La regresión logística optimizada mejora el F1-score respecto al modelo base.  
- Se logró superar el **criterio de éxito ≥ 5% de mejora** (según resultados de Grid/Random Search).  
- El modelo conserva interpretabilidad, baja complejidad y tiempo mínimo de inferencia.  
- No se observan trade-offs negativos significativos.

Por ello, **la regresión logística optimizada se mantiene como el modelo final del proyecto**.
